# 06 Random Forest Modelling

This notebook trains and evaluates a Random Forest classifier using the current Dublin Bikes feature set and the current `band_30min` target definition.

In [1]:
import torch
import time


print("torch:", torch.__version__)
print("cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("archs:", torch.cuda.get_arch_list())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

x = torch.rand(3000, 3000, device=device)
y = torch.rand(3000, 3000, device=device)

torch.cuda.synchronize()
t0 = time.time()
z = x @ y
torch.cuda.synchronize()
t1 = time.time()

print("Result device:", z.device)
print("Elapsed:", t1 - t0, "seconds")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED) 
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score

torch: 2.10.0+cu128
cuda runtime: 12.8
cuda available: True
archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
gpu: NVIDIA GeForce RTX 5060 Laptop GPU
Using: cuda
Result device: cuda:0
Elapsed: 0.24260187149047852 seconds


In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

PROJECT_ROOT = Path("..").resolve()
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_v2.parquet"
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_v2.csv.gz"

if FEATURES_PATH.suffix == ".parquet":
    df = pd.read_parquet(FEATURES_PATH).copy()
else:
    df = pd.read_csv(FEATURES_PATH).copy()

df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

print("Loaded:", FEATURES_PATH)
print("Columns:", df.columns.tolist())
print(df.shape)
df.head()


Loaded: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/dublin_bikes_features_v2.parquet
Columns: ['station_id', 'name', 'last_reported', 'hour_slot', 'hour_sin', 'hour_cos', 'days_of_week', 'is_weekday', 'roll_mean_30min', 'roll_std_30min', 'utilisation', 'dock_ratio', 'util_slope', 'util_vs_mean', 'time_slot_str', 'empty_risk', 'full_risk', 'center_index', 'area_type_str']
(13513883, 19)


,station_id,name,last_reported,hour_slot,hour_sin,hour_cos,days_of_week,is_weekday,roll_mean_30min,roll_std_30min,utilisation,dock_ratio,util_slope,util_vs_mean,time_slot_str,empty_risk,full_risk,center_index,area_type_str
0,1,CLARENDON ROW,2022-04-27 13:00:02,13,-0.258819,-0.965926,2,True,0.0,0.0,0.0,0.870968,NaN,0.0,13:00,True,False,1,retail
1,1,CLARENDON ROW,2022-04-27 13:30:02,13,-0.382683,-0.923880,2,True,0.0,0.0,0.0,0.870968,0.0,0.0,13:30,True,False,1,retail
2,1,CLARENDON ROW,2022-04-27 14:00:02,14,-0.500000,-0.866025,2,True,0.0,0.0,0.0,0.870968,0.0,0.0,14:00,True,False,1,retail
3,1,CLARENDON ROW,2022-04-27 14:30:02,14,-0.608761,-0.793353,2,True,0.0,0.0,0.0,0.000000,0.0,0.0,14:30,True,False,1,retail
4,1,CLARENDON ROW,2022-04-27 15:00:02,15,-0.707107,-0.707107,2,True,0.0,0.0,0.0,0.000000,0.0,0.0,15:00,True,False,1,retail


## Target creation

In [6]:

# Create the 30-minute-ahead target using the next available observation at or after t + 30 min
def get_target(group):
    group = group.sort_values("last_reported").copy()
    sid = group["station_id"].iloc[0]

    times = group["last_reported"].to_numpy()
    values = group["utilisation"].to_numpy()

    target = np.full(len(group), np.nan)

    for i in range(len(times)):
        t_future = times[i] + np.timedelta64(30, "m")
        idx = np.searchsorted(times, t_future, side="left")
        if idx < len(times):
            target[i] = values[idx]

    group["utilisation_tplus_30"] = target
    group["station_id"] = sid
    return group

parts = []
for _, g in df.groupby("station_id", sort=False):
    parts.append(get_target(g))

df = pd.concat(parts, ignore_index=True)

def utilisation_band(u):
    if u < 0.20:
        return 0
    elif u < 0.40:
        return 1
    elif u < 0.60:
        return 2
    elif u < 0.80:
        return 3
    else:
        return 4

df["band_30min"] = df["utilisation_tplus_30"].apply(utilisation_band)
band_labels = [0, 1, 2, 3, 4]

print("Band distribution:")
print(df["band_30min"].value_counts(normalize=True).sort_index())




Band distribution:
band_30min
0    0.348512
1    0.214146
2    0.172572
3    0.125730
4    0.139039
Name: proportion, dtype: float64


In [15]:

import numpy as np
import pandas as pd

df = df.sort_values(["station_id", "last_reported"]).copy()

exact_count = 0
later_count = 0
missing_count = 0

per_station_stats = []

for sid, g in df.groupby("station_id", sort=False):
    g = g.sort_values("last_reported").copy()
    times = g["last_reported"].to_numpy()

    station_exact = 0
    station_later = 0
    station_missing = 0

    for t in times:
        target_time = t + np.timedelta64(45, "m")
        idx = np.searchsorted(times, target_time, side="left")

        if idx >= len(times):
            station_missing += 1
        else:
            if times[idx] == target_time:
                station_exact += 1
            else:
                station_later += 1

    n = len(g)
    exact_count += station_exact
    later_count += station_later
    missing_count += station_missing

    per_station_stats.append({
        "station_id": sid,
        "n_obs": n,
        "exact_tplus30_pct": station_exact / n * 100,
        "later_than_tplus30_pct": station_later / n * 100,
        "missing_tplus30_or_later_pct": station_missing / n * 100
    })

total = exact_count + later_count + missing_count

print(f"Total observations: {total}")
print(f"Exact t+30 exists: {exact_count} ({exact_count / total:.4%})")
print(f"No exact t+30, but later t' >= t+30 exists: {later_count} ({later_count / total:.4%})")
print(f"No t' >= t+30 exists: {missing_count} ({missing_count / total:.4%})")

per_station_stats = pd.DataFrame(per_station_stats)
print(per_station_stats.head())

Total observations: 13509818
Exact t+30 exists: 4088799 (30.2654%)
No exact t+30, but later t' >= t+30 exists: 9420405 (69.7301%)
No t' >= t+30 exists: 614 (0.0045%)
   station_id   n_obs  exact_tplus30_pct  later_than_tplus30_pct  \
0           1  119501          39.764521               60.230458   
1           2  114148          23.450258               76.545362   
2           3  120264          32.609925               67.385918   
3           4  115138          25.896750               74.098039   
4           5  132648          47.004101               52.991376   

   missing_tplus30_or_later_pct  
0                      0.005021  
1                      0.004380  
2                      0.004158  
3                      0.005211  
4                      0.004523  


## Cleaning

In [11]:
missing = df["utilisation_tplus_30"].isna().sum()
total = len(df)

percentage = missing / total * 100

print(f"Missing targets: {missing}")
print(f"Total rows: {total}")
print(f"Percentage missing: {percentage:.4f}%")

missing_per_station = (
    df.groupby("station_id")["utilisation_tplus_30"]
      .apply(lambda x: x.isna().mean() * 100)
)

print("Missing target percentage per station:")
print(missing_per_station.sort_values(ascending=False).head(10))

df = df.dropna(subset=["utilisation_tplus_30", "util_slope", "util_vs_mean"]).copy()
print(df.shape)


Missing targets: 431
Total rows: 13513883
Percentage missing: 0.0032%
Missing target percentage per station:
station_id
70     0.012641
93     0.004490
84     0.004425
85     0.004392
83     0.004274
101    0.004272
71     0.004134
42     0.004069
104    0.003771
94     0.003683
Name: utilisation_tplus_30, dtype: float64
(13509818, 21)


## Features and time-based split

In [5]:

features = [
    "utilisation",
    # "roll_mean_30min", #remove due to high correlation with utilisation
    "roll_std_30min",
    # "dock_ratio", #remove due to high correlation with utilisation
    # "hour_slot",
    "hour_sin", 
    "hour_cos",
    "is_weekday",
    "center_index",
    'area_type_str',
    "util_slope", # trend of utilisation in last 30 min
    "util_vs_mean", # utilisation vs mean of station in last 30 min
    # "station_cluster"   # if added after Maxime's clustering
]
X = df[features].copy()
X = pd.get_dummies(X, columns=["area_type_str"], drop_first=True)

cutoff = df["last_reported"].max() - pd.Timedelta(days=120)
train_mask = df["last_reported"] < cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = df["band_30min"][train_mask], df["band_30min"][~train_mask]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).sort_index())


Train shape: (11085384, 10) Test shape: (2424434, 10)
Train target distribution:
band_30min
0    0.348777
1    0.214627
2    0.172247
3    0.125196
4    0.139152
Name: proportion, dtype: float64


In [6]:
print(X_train.columns.tolist())

['utilisation', 'roll_std_30min', 'hour_sin', 'hour_cos', 'is_weekday', 'center_index', 'util_slope', 'util_vs_mean', 'area_type_str_retail', 'area_type_str_work']


## Random Forest model

In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)
print("Training complete.")


## Evaluation

In [ ]:

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf, labels=band_labels))


In [ ]:

cm = confusion_matrix(y_test, y_pred_rf, labels=band_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=band_labels,
    yticklabels=band_labels
)
plt.title("Random Forest — Confusion Matrix")
plt.ylabel("True Band")
plt.xlabel("Predicted Band")
plt.tight_layout()
plt.show()


In [ ]:

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances.sort_values().plot(kind="barh", figsize=(8, 6), color="steelblue")
plt.title("Random Forest — Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()
